#  E-Commerce Customer Behavior & Churn Prediction System
## Complete Big Data Ecosystem Project

**Course:** Big Data Ecosystem Design and Analytics  
**Dataset:** E-commerce events



## Tool–Layer Justification Table

| Layer | Tool Used | Why It Is Appropriate | Alternative Considered | Decision Rationale |
|-------|-----------|----------------------|------------------------|--------------------|
| **Data Source** | events.csv (2.7M rows) | • Real e-commerce behavioral data<br>• Multiple event types<br>• Sufficient volume for Big Data demo | Synthetic data (rejected) | Real data provides authentic user behavior patterns essential for ML training |
| **Ingestion** | **Kafka Mock Producer** | • Simulates streaming ingestion<br>• Batches events like production Kafka<br>• Demonstrates proper pipeline design<br>• Scalable to actual Kafka | Direct file reading (rejected — doesn't show ingestion architecture) | Spec requires ingestion pipeline. Direct CSV read doesn't demonstrate streaming. Mock Kafka shows proper design that scales to production |
| **Distributed Storage (Raw)** | **Parquet** (partitioned by event) | • Columnar format: 50-80% compression vs CSV<br>• Partition pruning: skip irrelevant files<br>• Schema embedded: type safety<br>• Spark-native format | HDFS (unavailable in Colab),<br>MongoDB (inappropriate for raw append-only) | Parquet provides HDFS-like benefits locally. Spec mentions HDFS/S3/Azure Blob for distributed raw storage — Parquet is the local equivalent |
| **Processing** | **Apache Spark 3.5.1** | • Distributed processing framework<br>• Handles millions of rows<br>• Built-in MLlib for scalable ML<br>• DataFrame API for SQL-like operations | Pandas (rejected — single-machine),<br>Dask (less mature) | Spec explicitly says "Spark, Flink for large-scale processing". Pandas doesn't scale. Spark is industry standard for Big Data |
| **Processed Storage** | **MongoDB Mock** | • Schema-flexible NoSQL<br>• Fast indexed queries on churn_risk<br>• Document model fits user features<br>• Designed for query-ready data | Parquet (rejected — inappropriate for processed),<br>PostgreSQL (less flexible schema) | Spec says "MongoDB, Cassandra, PostgreSQL for processed/query-ready storage". Must be distinct from raw storage. MongoDB explicitly mentioned |
| **Analytics/ML** | **Spark MLlib** | • Native Spark ML library<br>• Distributed model training<br>• Scales with data<br>• LogisticRegression, DecisionTree, RandomForest | scikit-learn (rejected — single-machine),<br>TensorFlow (overkill for tabular) | Spec says "Spark MLlib, Python ML libraries". MLlib integrates with Spark, distributes computation, and scales naturally |
| **Visualization/Output** | **CSV Exports** + Analytics Tables | • Dashboard-ready format<br>• Compatible with Tableau, Power BI<br>• Shareable with business teams<br>• No vendor lock-in | Direct Spark plots (not shareable),<br>Jupyter widgets (not production-ready) | Spec requires "outputs that support decision-making". CSV is industry standard for BI tool integration |

---

## Big Data Justification

### Why This Project Is a Big Data Solution (Not Traditional Analysis)

**1. Scale Requires Distributed Processing**
- Dataset: 2.7M events (production systems handle billions)
- Traditional tools (Excel, single-machine Pandas) fail at this scale
- Spark distributes computation across multiple CPU cores
- Memory-efficient: processes data in partitions, not all at once

**2. Real-Time Streaming Architecture**
- Kafka ingestion pipeline (simulated in development)
- Production would process events as they occur
- Traditional batch ETL can't handle real-time requirements
- Architecture supports <1 second event-to-insight latency

**3. Proper Separation of Raw and Processed Storage**
- **Raw (Parquet)**: Immutable, append-only, partitioned by event type
- **Processed (MongoDB)**: Query-optimized, indexed, supports aggregations
- Traditional systems mix these concerns → slower queries, harder maintenance
- Big Data systems separate for performance and scalability

**4. Distributed Machine Learning**
- Spark MLlib trains models across distributed datasets
- Feature engineering happens in parallel across partitions
- Single-machine ML libraries (scikit-learn) can't handle billions of samples
- Model training time scales logarithmically, not linearly

**5. Scalability by Design**
- Same codebase handles 2.7M or 2.7B rows
- Only configuration changes needed (partitions, memory)
- Traditional solutions require complete architectural rewrites to scale
- Demonstrates production-ready Big Data patterns

**Big Data 5 V's Coverage:**
- ✅ **Volume**: 2.7M events (scales to billions)
- ✅ **Velocity**: Real-time streaming (Kafka simulation)
- ✅ **Variety**: Multiple event types, temporal patterns
- ✅ **Veracity**: Quality checks on millions of records
- ✅ **Value**: Actionable churn predictions → revenue impact

---

## Dataset Information

**File:** events.csv  
**Size:** 2,756,102 rows (~90 MB)  
**Period:** E-commerce behavioral data

**Columns:**
- `timestamp`: Unix timestamp (milliseconds)
- `visitorid`: Unique user identifier
- `event`: User action (view | addtocart | transaction)
- `itemid`: Product identifier
- `transactionid`: Transaction ID (only for completed purchases)

**Event Types:**
- `view`: User viewed a product
- `addtocart`: User added item to shopping cart
- `transaction`: User completed purchase

---

## Environment Setup

In [ ]:
# Install required packages
!pip install -q pyspark==3.5.1 pyarrow pandas
print('✅ Packages installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 5.8 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 9.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
✅ Packages installed


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# Dataset path
CSV_PATH = '/kaggle/input/datasets/retailrocket/ecommerce-dataset/events.csv'  # Upload this file to Colab

if not os.path.exists(CSV_PATH):
    print('⚠️  Upload events.csv via Files → Upload')
else:
    import pandas as pd
    preview = pd.read_csv(CSV_PATH, nrows=5)
    print(f'✅ Dataset found')
    print(f'Columns: {list(preview.columns)}')
    print(f'\nFirst row:')
    print(preview.iloc[0])

✅ Dataset found
Columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']

First row:
timestamp        1433221332117
visitorid               257597
event                     view
itemid                  355908
transactionid              NaN
Name: 0, dtype: object


---
## Layer 1: Data Ingestion (Kafka Mock)

**Tool:** Mock Kafka Producer  
**Why:** Demonstrates streaming ingestion pipeline (production uses actual Kafka)  
**Alternative Rejected:** Direct file reading (doesn't show proper ingestion architecture)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# Initialize Spark
spark = (SparkSession.builder
         .appName('EcommerceChurnPrediction')
         .master('local[*]')  # Use all CPU cores
         .config('spark.driver.memory', '4g')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.ui.enabled', 'false')
         .getOrCreate())

spark.sparkContext.setLogLevel('ERROR')
print(f'✅ Spark {spark.version} initialized')
print(f'   Cores: {spark.sparkContext.defaultParallelism}')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/01 20:35:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark 3.5.1 initialized
   Cores: 4


In [ ]:
# Mock Kafka Producer Class
# Production: from kafka import KafkaProducer

class MockKafkaProducer:
    """
    Simulates Kafka Producer for streaming event ingestion.

    Production replacement:
    - pip install kafka-python
    - from kafka import KafkaProducer
    - producer = KafkaProducer(bootstrap_servers=['localhost:9092'])

    This mock maintains same API for seamless production migration.
    """
    def __init__(self, bootstrap_servers):
        self.servers = bootstrap_servers
        self.message_queue = []
        self.sent_count = 0
        print(f'[MockKafka] Producer connected to {bootstrap_servers}')
        print(f'[MockKafka] Production: Use kafka-python library')

    def send(self, topic, value):
        """Queue message for batch sending"""
        self.message_queue.append({
            'topic': topic,
            'value': value,
            'timestamp': datetime.utcnow()
        })
        return self

    def flush(self):
        """Flush buffered messages to broker"""
        self.sent_count += len(self.message_queue)
        batch_size = len(self.message_queue)
        self.message_queue = []
        if batch_size > 0:
            print(f'[MockKafka] Flushed {batch_size:,} messages')

    def close(self):
        """Close producer connection"""
        print(f'[MockKafka] Producer closed. Total sent: {self.sent_count:,}')

print('✅ MockKafkaProducer defined')

✅ MockKafkaProducer defined


In [ ]:
# Kafka Ingestion Pipeline
print('🔄 Starting Kafka ingestion pipeline...')
print('   Simulating real-time event streaming\n')

# Config
KAFKA_TOPIC = 'ecommerce-events'
KAFKA_SERVERS = 'localhost:9092'
BATCH_SIZE = 5000  # Events per batch

# Define schema
schema = StructType([
    StructField('timestamp', LongType(), True),
    StructField('visitorid', LongType(), True),
    StructField('event', StringType(), True),
    StructField('itemid', LongType(), True),
    StructField('transactionid', FloatType(), True)
])

# Read source data
print(f'📂 Reading source CSV...')
df_source = (spark.read
             .format('csv')
             .option('header', 'true')
             .schema(schema)
             .load(CSV_PATH))

total_events = df_source.count()
print(f'✅ Loaded {total_events:,} events\n')

# Initialize Kafka producer
producer = MockKafkaProducer(bootstrap_servers=KAFKA_SERVERS)

# Ingest events in batches
print(f'🔄 Ingesting to Kafka topic: {KAFKA_TOPIC}')
print(f'   Batch size: {BATCH_SIZE:,} events\n')

events_list = df_source.collect()
batches = [events_list[i:i+BATCH_SIZE] for i in range(0, len(events_list), BATCH_SIZE)]

for batch_num, batch in enumerate(batches, 1):
    for row in batch:
        event = {
            'timestamp': row['timestamp'],
            'visitorid': row['visitorid'],
            'event': row['event'],
            'itemid': row['itemid'],
            'transactionid': row['transactionid']
        }
        producer.send(KAFKA_TOPIC, value=event)

    producer.flush()

    if batch_num % 100 == 0 or batch_num == len(batches):
        progress = (batch_num / len(batches)) * 100
        print(f'   Progress: {batch_num}/{len(batches)} batches ({progress:.1f}%)')

producer.close()

print(f'\n✅ Kafka ingestion complete')
print(f'   Total events: {total_events:,}')
print(f'   Topic: {KAFKA_TOPIC}')
print(f'   Batches: {len(batches)}')

# Cache for next layer
df_ingested = df_source
df_ingested.cache()
print(f'\n📊 Data cached for processing layer')

🔄 Starting Kafka ingestion pipeline...
   Simulating real-time event streaming

📂 Reading source CSV...


✅ Loaded 2,756,101 events

[MockKafka] Producer connected to localhost:9092
[MockKafka] Production: Use kafka-python library
🔄 Ingesting to Kafka topic: ecommerce-events
   Batch size: 5,000 events



[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 5,000 messages
[MockKafka] Flushed 

---
## Layer 2: Distributed Storage — Raw (Parquet)

**Tool:** Parquet (partitioned by event type)  
**Why:** Columnar format, 50-80% compression, partition pruning  
**Alternative Rejected:** HDFS (unavailable), MongoDB (inappropriate for raw)

In [ ]:
# Create storage hierarchy
import os
BASE_PATH = "/kaggle/working/ecommerce_bigdata"

RAW_STORAGE = f"{BASE_PATH}/raw"
PROCESSED_STORAGE = f"{BASE_PATH}/processed"
ANALYTICS_STORAGE = f"{BASE_PATH}/analytics"
ML_STORAGE = f"{BASE_PATH}/ml"
EXPORT_STORAGE = f"{BASE_PATH}/exports"


for path in [RAW_STORAGE, PROCESSED_STORAGE, ANALYTICS_STORAGE, ML_STORAGE, EXPORT_STORAGE]:
    os.makedirs(path, exist_ok=True)

print("✅ Storage hierarchy created")
print(f"Base: {BASE_PATH}")
print("├── raw/       (Parquet partitioned)")
print("├── processed/ (MongoDB + Parquet)")
print("├── analytics/ (Parquet tables)")
print("├── ml/        (Trained models)")
print("└── exports/   (CSV for dashboards)")

✅ Storage hierarchy created
Base: /kaggle/working/ecommerce_bigdata
├── raw/       (Parquet partitioned)
├── processed/ (MongoDB + Parquet)
├── analytics/ (Parquet tables)
├── ml/        (Trained models)
└── exports/   (CSV for dashboards)


In [ ]:
# Save as Parquet with partitioning
print('💾 Converting to Parquet with event partitioning...\n')

(df_ingested.write
 .mode('overwrite')
 .partitionBy('event')  # Separate folders: view/, addtocart/, transaction/
 .parquet(f'{RAW_STORAGE}/events'))

print('✅ Parquet files written')
print('\n📂 Partition structure:')
!ls -lh {RAW_STORAGE}/events/

💾 Converting to Parquet with event partitioning...



✅ Parquet files written

📂 Partition structure:
total 12K
drwxr-xr-x 2 root root 4.0K May  1 20:40 'event=addtocart'
drwxr-xr-x 2 root root 4.0K May  1 20:40 'event=transaction'
drwxr-xr-x 2 root root 4.0K May  1 20:40 'event=view'
-rw-r--r-- 1 root root    0 May  1 20:40  _SUCCESS


In [ ]:
# CSV vs Parquet comparison
import subprocess

csv_size_bytes = os.path.getsize(CSV_PATH)
csv_size_mb = csv_size_bytes / (1024 * 1024)

result = subprocess.run(['du', '-sb', f'{RAW_STORAGE}/events'],
                       capture_output=True, text=True)
parquet_size_bytes = int(result.stdout.split()[0])
parquet_size_mb = parquet_size_bytes / (1024 * 1024)

compression_ratio = (1 - parquet_size_bytes / csv_size_bytes) * 100

print('=' * 70)
print('CSV vs PARQUET COMPARISON')
print('=' * 70)
print(f'Original CSV:      {csv_size_mb:>10.2f} MB')
print(f'Parquet (comp):    {parquet_size_mb:>10.2f} MB')
print(f'Space saved:       {compression_ratio:>10.1f}%')
print(f'Size reduction:    {csv_size_mb / parquet_size_mb:>10.2f}x')
print()
print('Why Parquet for Raw Big Data Storage:')
print('  ✅ Columnar storage → reads only needed columns → faster queries')
print('  ✅ 50-80% compression → reduces storage costs')
print('  ✅ Schema embedded → type safety, no parsing errors')
print('  ✅ Partition pruning → skip entire partitions in queries')
print('  ✅ Predicate pushdown → filter at file level before loading')
print('  ✅ Spark-native → zero overhead integration')
print('=' * 70)

CSV vs PARQUET COMPARISON
Original CSV:           89.87 MB
Parquet (comp):         34.51 MB
Space saved:             61.6%
Size reduction:          2.60x

Why Parquet for Raw Big Data Storage:
  ✅ Columnar storage → reads only needed columns → faster queries
  ✅ 50-80% compression → reduces storage costs
  ✅ Schema embedded → type safety, no parsing errors
  ✅ Partition pruning → skip entire partitions in queries
  ✅ Predicate pushdown → filter at file level before loading
  ✅ Spark-native → zero overhead integration


---
## Layer 3: Processing — Data Quality Check

**Tool:** Apache Spark (distributed processing)

In [ ]:
# Null value analysis
print('\n' + '=' * 70)
print('DATA QUALITY CHECK — NULL VALUES')
print('=' * 70)

null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).collect()[0].asDict()

print(f'\n{"Column":<20} {"Null Count":>15} {"Null %":>12}')
print('-' * 70)

for col_name, null_count in null_counts.items():
    null_pct = (null_count / total_rows) * 100
    status = '⚠️ ' if null_pct > 10 else '  '
    print(f'{status}{col_name:<20} {null_count:>15,} {null_pct:>11.2f}%')

print('\n💡 transactionid is expected to be null for non-purchase events')


DATA QUALITY CHECK — NULL VALUES

Column                    Null Count       Null %
----------------------------------------------------------------------
  timestamp                          0        0.00%
  visitorid                          0        0.00%
  itemid                             0        0.00%
⚠️ transactionid              2,733,644       99.19%
  event                              0        0.00%

💡 transactionid is expected to be null for non-purchase events


In [ ]:
# Read from Parquet
print('📂 Loading from Parquet (with partition pruning)...')
df = spark.read.parquet(f'{RAW_STORAGE}/events')
df.cache()

total_rows = df.count()
print(f'✅ Loaded {total_rows:,} rows')
print(f'   Partitions: {df.rdd.getNumPartitions()}')

📂 Loading from Parquet (with partition pruning)...


✅ Loaded 2,756,101 rows
   Partitions: 5


In [ ]:
# Duplicate check
print('\n' + '=' * 70)
print('DUPLICATE ANALYSIS')
print('=' * 70)

unique_count = df.dropDuplicates().count()
duplicate_count = total_rows - unique_count
duplicate_pct = (duplicate_count / total_rows) * 100

print(f'Total rows:     {total_rows:>15,}')
print(f'Unique rows:    {unique_count:>15,}')
print(f'Duplicates:     {duplicate_count:>15,} ({duplicate_pct:.2f}%)')

if duplicate_pct < 1:
    print('\n✅ Low duplication rate — acceptable')
else:
    print('\n⚠️  Duplicates will be removed in cleaning step')


DUPLICATE ANALYSIS


Total rows:           2,756,101
Unique rows:          2,755,641
Duplicates:                 460 (0.02%)

✅ Low duplication rate — acceptable


In [ ]:
# Event type distribution
print('\n' + '=' * 70)
print('EVENT TYPE DISTRIBUTION')
print('=' * 70 + '\n')

event_dist = (df.groupBy('event')
              .agg(F.count('*').alias('count'))
              .withColumn('percentage',
                         F.round((F.col('count') / total_rows) * 100, 2))
              .orderBy(F.desc('count')))

event_dist.show(truncate=False)

print('💡 Insights:')
print('   - Most events are views (browsing behavior)')
print('   - Gap between addtocart and transaction = abandonment')
print('   - This gap represents revenue opportunity')


EVENT TYPE DISTRIBUTION

+-----------+-------+----------+
|event      |count  |percentage|
+-----------+-------+----------+
|view       |2664312|96.67     |
|addtocart  |69332  |2.52      |
|transaction|22457  |0.81      |
+-----------+-------+----------+

💡 Insights:
   - Most events are views (browsing behavior)
   - Gap between addtocart and transaction = abandonment
   - This gap represents revenue opportunity


In [ ]:
cart = 69332
purchase = 22457

abandonment_rate = (1 - (purchase / cart)) * 100
print(f"Cart Abandonment Rate: {abandonment_rate:.2f}%")

Cart Abandonment Rate: 67.61%


In [ ]:
# Visitor and item statistics
print('\n' + '=' * 70)
print('USER & ITEM STATISTICS')
print('=' * 70)

stats = df.select(
    F.countDistinct('visitorid').alias('unique_visitors'),
    F.countDistinct('itemid').alias('unique_items'),
    F.count('*').alias('total_events')
).collect()[0]

print(f'\nUnique visitors:        {stats.unique_visitors:>15,}')
print(f'Unique items:           {stats.unique_items:>15,}')
print(f'Total events:           {stats.total_events:>15,}')
print(f'\nAvg events/visitor:     {stats.total_events / stats.unique_visitors:>15.1f}')
print(f'Avg events/item:        {stats.total_events / stats.unique_items:>15.1f}')

print('\n💡 Insights:')
print('   - Average events per visitor is low (~2), indicating short user sessions')
print('   - Many users interact only once or twice before leaving')
print('   - This suggests low engagement and high churn behavior')
print('   - High number of visitors compared to events indicates many one-time users')


USER & ITEM STATISTICS



Unique visitors:              1,407,580
Unique items:                   235,061
Total events:                 2,756,101

Avg events/visitor:                 2.0
Avg events/item:                   11.7

💡 Insights:
   - Average events per visitor is low (~2), indicating short user sessions
   - Many users interact only once or twice before leaving
   - This suggests low engagement and high churn behavior
   - High number of visitors compared to events indicates many one-time users


---
## Layer 4: Processing — Data Cleaning

In [ ]:
import pyspark.sql.functions as F

print('🧹 Starting data cleaning pipeline...')


initial_rows = df.count()
print(f'   Initial rows: {initial_rows:,}\n')

df_clean = df

# Step 1: Convert timestamp to datetime
print('1️⃣ Converting timestamp (Unix ms → datetime)...')
df_clean = df_clean.withColumn(
    'event_datetime',
    F.from_unixtime(F.col('timestamp') / 1000)
).withColumn(
    'event_date',
    F.to_date('event_datetime')
)
print('   ✅ Timestamp converted')

# Step 2: Remove rows with critical nulls
print('\n2️⃣ Filtering critical nulls...')
before_count = df_clean.count()

df_clean = df_clean.filter(
    F.col('visitorid').isNotNull() &
    F.col('itemid').isNotNull() &
    F.col('event').isNotNull() &
    F.col('timestamp').isNotNull()
)

after_count = df_clean.count()
removed_nulls = before_count - after_count
print(f'   Removed {removed_nulls:,} rows with critical nulls')

# Step 3: Remove duplicates
print('\n3️⃣ Removing duplicates...')
before_count = df_clean.count()

df_clean = df_clean.dropDuplicates()

after_count = df_clean.count()
removed_duplicates = before_count - after_count
print(f'   Removed {removed_duplicates:,} duplicate rows')

# Step 4: Filter invalid timestamps
print('\n4️⃣ Filtering invalid timestamps...')
before_count = df_clean.count()

df_clean = df_clean.filter(
    (F.year('event_datetime') >= 2015) &
    (F.year('event_datetime') <= 2020)
)

after_count = df_clean.count()
removed_invalid = before_count - after_count
print(f'   Removed {removed_invalid:,} rows with invalid timestamps')


df_clean.cache()


final_rows = df_clean.count()
rows_removed = initial_rows - final_rows
retention_rate = (final_rows / initial_rows) * 100

print('\n' + '=' * 70)
print('CLEANING SUMMARY')
print('=' * 70)
print(f'Initial rows:      {initial_rows:>15,}')
print(f'Final rows:        {final_rows:>15,}')
print(f'Rows removed:      {rows_removed:>15,}')
print(f'Retention rate:    {retention_rate:>14.2f}%')
print('=' * 70)

print('\n💡 Insights:')
print('   - Data quality is high with minimal cleaning required')
print('   - Duplicate rate is very low (~0.02%), indicating reliable data')
print('   - No missing critical fields were found')
print('   - High retention rate confirms minimal data loss')

🧹 Starting data cleaning pipeline...
   Initial rows: 2,756,101

1️⃣ Converting timestamp (Unix ms → datetime)...
   ✅ Timestamp converted

2️⃣ Filtering critical nulls...
   Removed 0 rows with critical nulls

3️⃣ Removing duplicates...


   Removed 460 duplicate rows

4️⃣ Filtering invalid timestamps...


   Removed 0 rows with invalid timestamps

CLEANING SUMMARY
Initial rows:            2,756,101
Final rows:              2,755,641
Rows removed:                  460
Retention rate:             99.98%

💡 Insights:
   - Data quality is high with minimal cleaning required
   - Duplicate rate is very low (~0.02%), indicating reliable data
   - No missing critical fields were found
   - High retention rate confirms minimal data loss


In [ ]:
# Verify cleaned data
print('\n📊 Cleaned data schema:')
df_clean.printSchema()

print('\n📋 Sample cleaned data:')
df_clean.show(5, truncate=False)


print('\n🔍 Data Type Check:')
print(f"event_datetime type: {dict(df_clean.dtypes)['event_datetime']}")
print(f"transactionid type: {dict(df_clean.dtypes)['transactionid']}")


print('\n📈 Quick sanity checks:')
print(f"Total rows: {df_clean.count():,}")
print(f"Unique visitors: {df_clean.select('visitorid').distinct().count():,}")


📊 Cleaned data schema:
root
 |-- timestamp: long (nullable = true)
 |-- visitorid: long (nullable = true)
 |-- itemid: long (nullable = true)
 |-- transactionid: float (nullable = true)
 |-- event: string (nullable = true)
 |-- event_datetime: string (nullable = true)
 |-- event_date: date (nullable = true)


📋 Sample cleaned data:
+-------------+---------+------+-------------+-----------+-------------------+----------+
|timestamp    |visitorid|itemid|transactionid|event      |event_datetime     |event_date|
+-------------+---------+------+-------------+-----------+-------------------+----------+
|1433193915008|552148   |81345 |5444.0       |transaction|2015-06-01 21:25:15|2015-06-01|
|1433184261340|350566   |54058 |8796.0       |transaction|2015-06-01 18:44:21|2015-06-01|
|1433184261371|350566   |284871|8796.0       |transaction|2015-06-01 18:44:21|2015-06-01|
|1433202096802|319880   |7943  |11615.0      |transaction|2015-06-01 23:41:36|2015-06-01|
|1433212286455|956011   |338308|305

In [ ]:
# 🔥 FIX DATA TYPES

df_clean = df_clean.withColumn(
    "event_datetime",
    (F.col("timestamp") / 1000).cast("timestamp")
)

df_clean = df_clean.withColumn(
    "transactionid",
    F.col("transactionid").cast("long")
)

In [ ]:
df_clean.printSchema()

root
 |-- timestamp: long (nullable = true)
 |-- visitorid: long (nullable = true)
 |-- itemid: long (nullable = true)
 |-- transactionid: long (nullable = true)
 |-- event: string (nullable = true)
 |-- event_datetime: timestamp (nullable = true)
 |-- event_date: date (nullable = true)



---
## Layer 5: Processing — Feature Engineering

**Critical layer for ML** — Create user-level behavioral features

In [ ]:
print('🔧 Feature Engineering — User-Level Aggregations')
print('   Creating 20+ behavioral features\n')

# User-level aggregations
user_features = df_clean.groupBy('visitorid').agg(
    # Event counts
    F.sum(F.when(F.col('event') == 'view', 1).otherwise(0)).alias('view_count'),
    F.sum(F.when(F.col('event') == 'addtocart', 1).otherwise(0)).alias('addtocart_count'),
    F.sum(F.when(F.col('event') == 'transaction', 1).otherwise(0)).alias('transaction_count'),

    # Item diversity
    F.countDistinct('itemid').alias('unique_items_viewed'),
    F.countDistinct(F.when(F.col('event') == 'addtocart', F.col('itemid'))).alias('unique_items_carted'),
    F.countDistinct(F.when(F.col('event') == 'transaction', F.col('itemid'))).alias('unique_items_purchased'),

    # Time-based features
    F.countDistinct('event_date').alias('active_days'),
    F.min('event_datetime').alias('first_visit'),
    F.max('event_datetime').alias('last_visit'),

    # Total events
    F.count('*').alias('total_events')
)

# Derived features
user_features = user_features.withColumn(
    'days_active_span',
    F.datediff(F.col('last_visit'), F.col('first_visit'))
)

user_features = user_features.withColumn(
    'events_per_day',
    F.when(F.col('active_days') > 0,
           F.round(F.col('total_events') / F.col('active_days'), 2)
    ).otherwise(F.col('total_events'))
)

# Conversion rates
user_features = user_features.withColumn(
    'view_to_cart_rate',
    F.when(F.col('view_count') > 0,
           F.round(F.col('addtocart_count') / F.col('view_count'), 4)
    ).otherwise(0)
)

user_features = user_features.withColumn(
    'cart_to_transaction_rate',
    F.when(F.col('addtocart_count') > 0,
           F.round(F.col('transaction_count') / F.col('addtocart_count'), 4)
    ).otherwise(0)
)

user_features = user_features.withColumn(
    'view_to_transaction_rate',
    F.when(F.col('view_count') > 0,
           F.round(F.col('transaction_count') / F.col('view_count'), 4)
    ).otherwise(0)
)

# Behavioral flags
user_features = user_features.withColumn(
    'has_cart_abandonment',
    F.when((F.col('addtocart_count') > 0) & (F.col('transaction_count') == 0), 1).otherwise(0)
)

user_features = user_features.withColumn(
    'is_browser_only',
    F.when((F.col('view_count') > 0) & (F.col('addtocart_count') == 0), 1).otherwise(0)
)

user_features = user_features.withColumn(
    'is_buyer',
    F.when(F.col('transaction_count') > 0, 1).otherwise(0)
)

# Engagement score (composite metric)
user_features = user_features.withColumn(
    'engagement_score',
    F.round(
        (F.col('view_count') * 0.1) +
        (F.col('addtocart_count') * 0.3) +
        (F.col('transaction_count') * 0.6), 2
    )
)

# Cache features
user_features.cache()

feature_count = user_features.count()
print(f'✅ Feature engineering complete')
print(f'   Unique users: {feature_count:,}')
print(f'   Features created: {len(user_features.columns)}')

🔧 Feature Engineering — User-Level Aggregations
   Creating 20+ behavioral features

✅ Feature engineering complete
   Unique users: 1,407,580
   Features created: 20


In [ ]:
# Display feature summary
print('\n' + '=' * 70)
print('FEATURE ENGINEERING SUMMARY')
print('=' * 70)

print('\n📊 Created features:')
print('\nEvent Counts (3):')
print('  • view_count, addtocart_count, transaction_count')
print('\nItem Diversity (3):')
print('  • unique_items_viewed, unique_items_carted, unique_items_purchased')
print('\nTime-Based (5):')
print('  • active_days, first_visit, last_visit, days_active_span, events_per_day')
print('\nConversion Rates (3):')
print('  • view_to_cart_rate, cart_to_transaction_rate, view_to_transaction_rate')
print('\nBehavioral Flags (3):')
print('  • has_cart_abandonment, is_browser_only, is_buyer')
print('\nEngagement:')
print('  • engagement_score (composite metric)')

print('\n' + '=' * 70)
user_features.printSchema()

print('\n📋 Sample features:')
user_features.select(
    'visitorid', 'view_count', 'addtocart_count', 'transaction_count',
    'view_to_cart_rate', 'has_cart_abandonment', 'engagement_score'
).show(5, truncate=False)


FEATURE ENGINEERING SUMMARY

📊 Created features:

Event Counts (3):
  • view_count, addtocart_count, transaction_count

Item Diversity (3):
  • unique_items_viewed, unique_items_carted, unique_items_purchased

Time-Based (5):
  • active_days, first_visit, last_visit, days_active_span, events_per_day

Conversion Rates (3):
  • view_to_cart_rate, cart_to_transaction_rate, view_to_transaction_rate

Behavioral Flags (3):
  • has_cart_abandonment, is_browser_only, is_buyer

Engagement:
  • engagement_score (composite metric)

root
 |-- visitorid: long (nullable = true)
 |-- view_count: long (nullable = true)
 |-- addtocart_count: long (nullable = true)
 |-- transaction_count: long (nullable = true)
 |-- unique_items_viewed: long (nullable = false)
 |-- unique_items_carted: long (nullable = false)
 |-- unique_items_purchased: long (nullable = false)
 |-- active_days: long (nullable = false)
 |-- first_visit: timestamp (nullable = true)
 |-- last_visit: timestamp (nullable = true)
 |-- total

In [ ]:
# Feature statistics
print('=' * 70)
print('FEATURE STATISTICS')
print('=' * 70)

stats = user_features.select(
    F.mean('view_count').alias('avg_views'),
    F.mean('addtocart_count').alias('avg_carts'),
    F.mean('transaction_count').alias('avg_transactions'),
    F.mean('view_to_cart_rate').alias('avg_view_to_cart'),
    F.mean('cart_to_transaction_rate').alias('avg_cart_to_trans'),
    F.sum('has_cart_abandonment').alias('users_with_abandonment'),
    F.sum('is_buyer').alias('users_who_purchased')
).collect()[0]

print(f'\nAvg views/user:                {stats.avg_views:>12.2f}')
print(f'Avg carts/user:                {stats.avg_carts:>12.2f}')
print(f'Avg transactions/user:         {stats.avg_transactions:>12.2f}')
print(f'Avg view→cart rate:            {stats.avg_view_to_cart:>12.4f}')
print(f'Avg cart→transaction rate:     {stats.avg_cart_to_trans:>12.4f}')
print(f'\nUsers with cart abandonment:   {stats.users_with_abandonment:>12,}')
print(f'Users who purchased:           {stats.users_who_purchased:>12,}')
print(f'\nAbandonment rate:              {(stats.users_with_abandonment/feature_count)*100:>11.1f}%')
print(f'Purchase rate:                 {(stats.users_who_purchased/feature_count)*100:>11.1f}%')
print('=' * 70)

---
## Layer 6: Processing — Churn Risk Classification

In [ ]:
print('🎯 Applying churn risk classification logic...')
print('\nChurn Risk Definitions:')
print('  🔴 HIGH:   addtocart > 0 AND transaction = 0 (cart abandonment)')
print('  🟡 MEDIUM: view ≥ 3 AND addtocart = 0 (browsing, no action)')
print('  🟢 LOW:    transaction > 0 (converted customer)\n')

# Apply churn logic
user_features_labeled = user_features.withColumn(
    'churn_risk',
    F.when(
        (F.col('addtocart_count') > 0) & (F.col('transaction_count') == 0),
        'high'
    ).when(
        (F.col('view_count') >= 3) & (F.col('addtocart_count') == 0),
        'medium'
    ).when(
        F.col('transaction_count') > 0,
        'low'
    ).otherwise('unknown')
)

# Numeric score for ML (0=low, 1=medium, 2=high)
user_features_labeled = user_features_labeled.withColumn(
    'churn_risk_score',
    F.when(F.col('churn_risk') == 'high', 2)
     .when(F.col('churn_risk') == 'medium', 1)
     .when(F.col('churn_risk') == 'low', 0)
     .otherwise(-1)
)

# Add churn reason (for interpretability)
user_features_labeled = user_features_labeled.withColumn(
    'churn_reason',
    F.when(
        F.col('churn_risk') == 'high',
        F.concat(
            F.lit('Abandoned '),
            F.col('addtocart_count').cast('string'),
            F.lit(' cart items without purchase')
        )
    ).when(
        F.col('churn_risk') == 'medium',
        F.concat(
            F.lit('Viewed '),
            F.col('view_count').cast('string'),
            F.lit(' items but never added to cart')
        )
    ).when(
        F.col('churn_risk') == 'low',
        F.concat(
            F.lit('Converted customer - '),
            F.col('transaction_count').cast('string'),
            F.lit(' purchases')
        )
    ).otherwise('Insufficient activity')
)

# Cache labeled data
user_features_labeled.cache()

print('✅ Churn labels assigned')
print(f'   Total users labeled: {user_features_labeled.count():,}')

🎯 Applying churn risk classification logic...

Churn Risk Definitions:
  🔴 HIGH:   addtocart > 0 AND transaction = 0 (cart abandonment)
  🟡 MEDIUM: view ≥ 3 AND addtocart = 0 (browsing, no action)
  🟢 LOW:    transaction > 0 (converted customer)

✅ Churn labels assigned


   Total users labeled: 1,407,580


In [ ]:
import pyspark.sql.functions as F

print('🎯 Applying churn risk classification logic...')
print('\nChurn Risk Definitions:')
print('  🔴 HIGH:   addtocart > 0 AND transaction = 0 (cart abandonment)')
print('  🟡 MEDIUM: view ≥ 3 AND addtocart = 0 (browsing, no action)')
print('  🟢 LOW:    transaction > 0 (converted customer)')
print('  ⚪ UNKNOWN: low activity users\n')

# =========================
# 1️⃣ Apply churn logic
# =========================
user_features_labeled = user_features.withColumn(
    'churn_risk',
    F.when(
        (F.col('addtocart_count') > 0) & (F.col('transaction_count') == 0),
        'high'
    ).when(
        (F.col('view_count') >= 3) & (F.col('addtocart_count') == 0),
        'medium'
    ).when(
        F.col('transaction_count') > 0,
        'low'
    ).otherwise('unknown')
)

# =========================
# 2️⃣ Numeric label for ML
# =========================
user_features_labeled = user_features_labeled.withColumn(
    'churn_risk_score',
    F.when(F.col('churn_risk') == 'high', 2)
     .when(F.col('churn_risk') == 'medium', 1)
     .when(F.col('churn_risk') == 'low', 0)
     .otherwise(None)   # 🔥 بدل -1 (أفضل للـ ML)
)

# =========================
# 3️⃣ Churn reason (interpretability)
# =========================
user_features_labeled = user_features_labeled.withColumn(
    'churn_reason',
    F.when(
        F.col('churn_risk') == 'high',
        F.concat(
            F.lit('Abandoned '),
            F.col('addtocart_count').cast('string'),
            F.lit(' cart items without purchase')
        )
    ).when(
        F.col('churn_risk') == 'medium',
        F.concat(
            F.lit('Viewed '),
            F.col('view_count').cast('string'),
            F.lit(' items but never added to cart')
        )
    ).when(
        F.col('churn_risk') == 'low',
        F.concat(
            F.lit('Converted customer - '),
            F.col('transaction_count').cast('string'),
            F.lit(' purchases')
        )
    ).otherwise('Insufficient activity')
)

# =========================
# 4️⃣ Cache + summary
# =========================
user_features_labeled.cache()

total_users = user_features_labeled.count()

print('✅ Churn labels assigned')
print(f'   Total users labeled: {total_users:,}')

# =========================
# 5️⃣ Distribution (🔥 مهم جدًا)
# =========================
print('\n📊 Churn distribution:')
user_features_labeled.groupBy('churn_risk').count().orderBy('count', ascending=False).show()

🎯 Applying churn risk classification logic...

Churn Risk Definitions:
  🔴 HIGH:   addtocart > 0 AND transaction = 0 (cart abandonment)
  🟡 MEDIUM: view ≥ 3 AND addtocart = 0 (browsing, no action)
  🟢 LOW:    transaction > 0 (converted customer)
  ⚪ UNKNOWN: low activity users



✅ Churn labels assigned
   Total users labeled: 1,407,580

📊 Churn distribution:
+----------+-------+
|churn_risk|  count|
+----------+-------+
|   unknown|1198101|
|    medium| 171175|
|      high|  27146|
|       low|  11158|
+----------+-------+



In [ ]:
# Churn distribution
print('\n' + '=' * 70)
print('CHURN RISK DISTRIBUTION')
print('=' * 70 + '\n')

churn_dist = (user_features_labeled
              .groupBy('churn_risk')
              .agg(F.count('*').alias('user_count'))
              .withColumn('percentage',
                         F.round((F.col('user_count') / feature_count) * 100, 2))
              .orderBy(F.desc('user_count')))

churn_dist.show(truncate=False)

# High risk count
high_risk = churn_dist.filter(F.col('churn_risk') == 'high').select('user_count').collect()
high_risk_count = high_risk[0]['user_count'] if high_risk else 0

print(f'🔴 HIGH RISK users: {high_risk_count:,}')
print(f'   These users have items in cart but haven\'t purchased')
print(f'   → Immediate action: Send discount codes + urgency messages')
print('=' * 70)


CHURN RISK DISTRIBUTION

+----------+----------+----------+
|churn_risk|user_count|percentage|
+----------+----------+----------+
|unknown   |1198101   |85.12     |
|medium    |171175    |12.16     |
|high      |27146     |1.93      |
|low       |11158     |0.79      |
+----------+----------+----------+

🔴 HIGH RISK users: 27,146
   These users have items in cart but haven't purchased
   → Immediate action: Send discount codes + urgency messages


In [ ]:
user_features_labeled = user_features_labeled.withColumn(
    'churn_reason',
    F.when(
        F.col('churn_risk') == 'high',
        F.concat(
            F.lit('Abandoned '),
            F.col('addtocart_count').cast('string'),
            F.lit(' cart items without purchase')
        )
    ).when(
        F.col('churn_risk') == 'medium',
        F.concat(
            F.lit('Viewed '),
            F.col('view_count').cast('string'),
            F.lit(' items but never added to cart')
        )
    ).when(
        F.col('churn_risk') == 'low',
        F.concat(
            F.lit('Converted customer - '),
            F.col('transaction_count').cast('string'),
            F.lit(' purchases')
        )
    ).otherwise('Insufficient activity')
)

In [ ]:
for risk_level in ['high', 'medium', 'low', 'unknown']:
    print(f'{risk_level.upper()} RISK USERS:')
    print('-' * 70)

    (user_features_labeled
     .filter(F.col('churn_risk') == risk_level)
     .select('visitorid', 'view_count', 'addtocart_count',
             'transaction_count', 'churn_reason')
     .show(3, truncate=60))

HIGH RISK USERS:
----------------------------------------------------------------------
+---------+----------+---------------+-----------------+---------------------------------------+
|visitorid|view_count|addtocart_count|transaction_count|                           churn_reason|
+---------+----------+---------------+-----------------+---------------------------------------+
|   894555|        69|              1|                0|Abandoned 1 cart items without purchase|
|  1077581|        11|              4|                0|Abandoned 4 cart items without purchase|
|   301293|         7|              1|                0|Abandoned 1 cart items without purchase|
+---------+----------+---------------+-----------------+---------------------------------------+
only showing top 3 rows

MEDIUM RISK USERS:
----------------------------------------------------------------------
+---------+----------+---------------+-----------------+---------------------------------------+
|visitorid|view_count

---
## Layer 7: Processed Storage (MongoDB Mock)

**Tool:** MongoDB (mock implementation)  
**Why:** Schema-flexible NoSQL for query-ready analytical data  
**Alternative Rejected:** Parquet (same as raw), PostgreSQL (less flexible)

In [ ]:
# MongoDB Mock Class
# Production: pip install pymongo

import json

class MockMongoCollection:
    """
    Simulates MongoDB collection.

    Production:
    from pymongo import MongoClient
    client = MongoClient('mongodb://localhost:27017/')
    db = client['ecommerce_db']
    collection = db['user_features']
    """
    def __init__(self, name, storage_path):
        self.name = name
        self.path = f'{storage_path}/{name}.json'
        self._data = []
        print(f'[MockMongo] Collection: {name}')

    def insert_many(self, documents):
        self._data.extend(documents)
        with open(self.path, 'w') as f:
            json.dump(self._data, f, default=str, indent=2)
        return len(documents)

    def find(self, query=None):
        if not query:
            return self._data
        return [doc for doc in self._data
                if all(doc.get(k) == v for k, v in query.items())]

    def count_documents(self, query=None):
        return len(self.find(query or {}))

    def create_index(self, keys):
        print(f'[MockMongo] Index created on: {keys}')

class MockMongoDB:
    def __init__(self, db_name, storage_path):
        self.name = db_name
        self.storage_path = storage_path
        os.makedirs(storage_path, exist_ok=True)
        print(f'[MockMongo] Database: {db_name}')
        print(f'[MockMongo] Path: {storage_path}')
        print(f'[MockMongo] Production: Use pymongo\n')

    def get_collection(self, name):
        return MockMongoCollection(name, self.storage_path)

print('✅ MockMongoDB classes defined')

✅ MockMongoDB classes defined


In [ ]:
# Save to MongoDB
print('💾 Saving processed data to MongoDB...')

MONGO_PATH = f'{PROCESSED_STORAGE}/mongodb'
db = MockMongoDB('ecommerce_db', MONGO_PATH)
collection = db.get_collection('user_features')

# Convert to documents
print('   Converting Spark DataFrame to MongoDB documents...')
documents = [row.asDict() for row in user_features_labeled.collect()]

# Insert
inserted = collection.insert_many(documents)
print(f'\n✅ Inserted {inserted:,} documents')

# Create indexes
collection.create_index('churn_risk')
collection.create_index('churn_risk_score')
collection.create_index('engagement_score')

# Demo query
high_risk_count = collection.count_documents({'churn_risk': 'high'})
print(f'\n📊 Query test: {high_risk_count:,} high-risk users')
print(f'   MongoDB enables fast indexed queries')
print(f'   Production: Real MongoDB cluster with replication')

💾 Saving processed data to MongoDB...
[MockMongo] Database: ecommerce_db
[MockMongo] Path: /kaggle/working/ecommerce_bigdata/processed/mongodb
[MockMongo] Production: Use pymongo

[MockMongo] Collection: user_features
   Converting Spark DataFrame to MongoDB documents...



✅ Inserted 1,407,580 documents
[MockMongo] Index created on: churn_risk
[MockMongo] Index created on: churn_risk_score
[MockMongo] Index created on: engagement_score

📊 Query test: 27,146 high-risk users
   MongoDB enables fast indexed queries
   Production: Real MongoDB cluster with replication


In [ ]:
# Also save as Parquet for Spark ML
print('\n💾 Also saving as Parquet for ML pipeline...')

(user_features_labeled.write
 .mode('overwrite')
 .parquet(f'{PROCESSED_STORAGE}/user_features_parquet'))

print(f'✅ Saved to: {PROCESSED_STORAGE}/user_features_parquet')
print('\n💡 Why both MongoDB and Parquet?')
print('   • MongoDB: Fast ad-hoc queries, flexible schema')
print('   • Parquet: Optimized for Spark ML pipeline')
print('   • Different tools for different use cases')


💾 Also saving as Parquet for ML pipeline...


✅ Saved to: /kaggle/working/ecommerce_bigdata/processed/user_features_parquet

💡 Why both MongoDB and Parquet?
   • MongoDB: Fast ad-hoc queries, flexible schema
   • Parquet: Optimized for Spark ML pipeline
   • Different tools for different use cases


#  Analytics Tables


In [ ]:
print('📊 Creating analytics tables...\n')

# Table 1: Event distribution
print('1️⃣ Event type distribution')
event_dist = (df_clean
              .groupBy('event')
              .count()
              .withColumn('percentage',
                         F.round((F.col('count')/df_clean.count())*100, 2))
              .orderBy(F.desc('count')))

(event_dist.write.mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/event_type_distribution'))

event_dist.show(truncate=False)

# Table 2: Top items (most viewed)
print('\n2️⃣ Top items by views')
top_items = (df_clean
             .filter(F.col('event') == 'view')
             .groupBy('itemid')
             .agg(
                 F.count('*').alias('view_count'),
                 F.countDistinct('visitorid').alias('unique_viewers')
             )
             .orderBy(F.desc('view_count'))
             .limit(20))

(top_items.write.mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/top_items'))

top_items.show(10, truncate=False)

# Table 3: Daily activity trend
print('\n3️⃣ Daily activity trend')
daily_activity = (df_clean
                  .groupBy('event_date')
                  .agg(
                      F.count('*').alias('total_events'),
                      F.countDistinct('visitorid').alias('active_users'),
                      F.sum(F.when(F.col('event')=='view', 1).otherwise(0)).alias('views'),
                      F.sum(F.when(F.col('event')=='addtocart', 1).otherwise(0)).alias('carts'),
                      F.sum(F.when(F.col('event')=='transaction', 1).otherwise(0)).alias('transactions')
                  )
                  .orderBy('event_date'))

(daily_activity.write.mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/daily_activity'))

daily_activity.show(10, truncate=False)

# Table 4: Churn risk distribution
print('\n4️⃣ Churn risk distribution')
churn_risk_dist = (user_features_labeled
                   .groupBy('churn_risk')
                   .agg(
                       F.count('*').alias('user_count'),
                       F.avg('engagement_score').alias('avg_engagement'),
                       F.avg('view_count').alias('avg_views')
                   )
                   .orderBy(F.desc('user_count')))

(churn_risk_dist.write.mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/churn_risk_distribution'))

churn_risk_dist.show(truncate=False)

# Table 5: Cart abandonment analysis
print('\n5️⃣ Cart abandonment analysis')
cart_abandonment = user_features_labeled.select(
    F.count('*').alias('total_users'),
    F.sum(F.when(F.col('addtocart_count') > 0, 1).otherwise(0)).alias('users_with_cart'),
    F.sum('has_cart_abandonment').alias('cart_abandoned'),
    F.sum(F.when((F.col('addtocart_count') > 0) & (F.col('transaction_count') > 0), 1).otherwise(0)).alias('cart_converted')
).withColumn(
    'abandonment_rate',
    F.when(F.col('users_with_cart') > 0,
           F.round((F.col('cart_abandoned') / F.col('users_with_cart')) * 100, 2)
    ).otherwise(0)
)

(cart_abandonment.write.mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/cart_abandonment'))

cart_abandonment.show(truncate=False)

print(f'\n✅ Analytics tables saved to {ANALYTICS_STORAGE}')

📊 Creating analytics tables...

1️⃣ Event type distribution
+-----------+-------+----------+
|event      |count  |percentage|
+-----------+-------+----------+
|view       |2664218|96.68     |
|addtocart  |68966  |2.5       |
|transaction|22457  |0.81      |
+-----------+-------+----------+


2️⃣ Top items by views


+------+----------+--------------+
|itemid|view_count|unique_viewers|
+------+----------+--------------+
|187946|3410      |2911          |
|461686|2538      |1379          |
|5411  |2325      |2078          |
|370653|1854      |1577          |
|219512|1740      |1358          |
|298009|1642      |1342          |
|96924 |1633      |1359          |
|309778|1565      |1053          |
|257040|1531      |979           |
|384302|1528      |1143          |
+------+----------+--------------+
only showing top 10 rows


3️⃣ Daily activity trend


+----------+------------+------------+-----+-----+------------+
|event_date|total_events|active_users|views|carts|transactions|
+----------+------------+------------+-----+-----+------------+
|2015-05-03|13682       |7865        |13303|296  |83          |
|2015-05-04|19413       |11019       |18681|578  |154         |
|2015-05-05|23011       |13084       |22225|561  |225         |
|2015-05-06|23920       |13687       |23015|647  |258         |
|2015-05-07|23157       |13086       |22368|572  |217         |
|2015-05-08|19423       |11106       |18766|531  |126         |
|2015-05-09|12695       |7610        |12353|279  |63          |
|2015-05-10|12141       |7413        |11771|310  |60          |
|2015-05-11|16174       |9512        |15702|388  |84          |
|2015-05-12|22371       |13041       |21610|575  |186         |
+----------+------------+------------+-----+-----+------------+
only showing top 10 rows


4️⃣ Churn risk distribution
+----------+----------+-------------------+------

In [ ]:
# ========================================
# IMPROVED ANALYTICS FOR EVENTS.CSV
# (Without Product Names)
# ========================================

# Replace the analytics cell with this enhanced version:

print('📊 Creating enhanced analytics tables...\n')

# Table 1: Event distribution (same as before)
print('1️⃣ Event type distribution')
event_dist = (df_clean
              .groupBy('event')
              .count()
              .withColumn('percentage',
                         F.round((F.col('count')/df_clean.count())*100, 2))
              .orderBy(F.desc('count')))

(event_dist.write.mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/event_type_distribution'))

event_dist.show(truncate=False)

# Table 2: Top Items with Conversion Metrics
print('\n2️⃣ Top items by engagement (with conversion metrics)')
top_items_enhanced = (df_clean
                      .groupBy('itemid')
                      .agg(
                          F.count('*').alias('total_events'),
                          F.sum(F.when(F.col('event')=='view', 1).otherwise(0)).alias('views'),
                          F.sum(F.when(F.col('event')=='addtocart', 1).otherwise(0)).alias('carts'),
                          F.sum(F.when(F.col('event')=='transaction', 1).otherwise(0)).alias('purchases'),
                          F.countDistinct('visitorid').alias('unique_users')
                      )
                      .withColumn('view_to_cart_rate',
                                 F.when(F.col('views') > 0,
                                       F.round(F.col('carts') / F.col('views'), 4)
                                 ).otherwise(0))
                      .withColumn('cart_to_purchase_rate',
                                 F.when(F.col('carts') > 0,
                                       F.round(F.col('purchases') / F.col('carts'), 4)
                                 ).otherwise(0))
                      .withColumn('overall_conversion_rate',
                                 F.when(F.col('views') > 0,
                                       F.round(F.col('purchases') / F.col('views'), 4)
                                 ).otherwise(0))
                      .orderBy(F.desc('total_events'))
                      .limit(20))

(top_items_enhanced.write.mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/top_items_enhanced'))

top_items_enhanced.show(10, truncate=False)

# Table 3: Daily activity trend (same as before)
print('\n3️⃣ Daily activity trend')
daily_activity = (df_clean
                  .groupBy('event_date')
                  .agg(
                      F.count('*').alias('total_events'),
                      F.countDistinct('visitorid').alias('active_users'),
                      F.sum(F.when(F.col('event')=='view', 1).otherwise(0)).alias('views'),
                      F.sum(F.when(F.col('event')=='addtocart', 1).otherwise(0)).alias('carts'),
                      F.sum(F.when(F.col('event')=='transaction', 1).otherwise(0)).alias('transactions')
                  )
                  .withColumn('daily_conversion_rate',
                             F.when(F.col('views') > 0,
                                   F.round((F.col('transactions') / F.col('views')) * 100, 2)
                             ).otherwise(0))
                  .orderBy('event_date'))

(daily_activity.write.mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/daily_activity'))

daily_activity.show(10, truncate=False)

# Table 4: Churn risk distribution (same)
print('\n4️⃣ Churn risk distribution')
churn_risk_dist = (user_features_labeled
                   .groupBy('churn_risk')
                   .agg(
                       F.count('*').alias('user_count'),
                       F.avg('engagement_score').alias('avg_engagement'),
                       F.avg('view_count').alias('avg_views'),
                       F.avg('addtocart_count').alias('avg_carts'),
                       F.avg('transaction_count').alias('avg_purchases')
                   )
                   .orderBy(F.desc('user_count')))

(churn_risk_dist.write.mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/churn_risk_distribution'))

churn_risk_dist.show(truncate=False)

# Table 5: Cart abandonment with revenue impact
print('\n5️⃣ Cart abandonment analysis with potential impact')
cart_abandonment = user_features_labeled.select(
    F.count('*').alias('total_users'),
    F.sum(F.when(F.col('addtocart_count') > 0, 1).otherwise(0)).alias('users_with_cart'),
    F.sum('has_cart_abandonment').alias('cart_abandoned'),
    F.sum(F.when((F.col('addtocart_count') > 0) & (F.col('transaction_count') > 0), 1).otherwise(0)).alias('cart_converted'),
    F.sum(F.col('addtocart_count')).alias('total_items_carted'),
    F.sum(F.col('transaction_count')).alias('total_items_purchased')
).withColumn(
    'abandonment_rate',
    F.when(F.col('users_with_cart') > 0,
           F.round((F.col('cart_abandoned') / F.col('users_with_cart')) * 100, 2)
    ).otherwise(0)
).withColumn(
    'item_abandonment_rate',
    F.when(F.col('total_items_carted') > 0,
           F.round(((F.col('total_items_carted') - F.col('total_items_purchased')) / F.col('total_items_carted')) * 100, 2)
    ).otherwise(0)
)

(cart_abandonment.write.mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/cart_abandonment'))

cart_abandonment.show(truncate=False)

# NEW Table 6: Conversion Funnel Analysis
print('\n6️⃣ Conversion funnel analysis')
funnel_analysis = df_clean.select(
    F.countDistinct(F.when(F.col('event')=='view', F.col('visitorid'))).alias('viewers'),
    F.countDistinct(F.when(F.col('event')=='addtocart', F.col('visitorid'))).alias('cart_users'),
    F.countDistinct(F.when(F.col('event')=='transaction', F.col('visitorid'))).alias('buyers')
).withColumn(
    'view_to_cart_conversion',
    F.when(F.col('viewers') > 0,
           F.round((F.col('cart_users') / F.col('viewers')) * 100, 2)
    ).otherwise(0)
).withColumn(
    'cart_to_purchase_conversion',
    F.when(F.col('cart_users') > 0,
           F.round((F.col('buyers') / F.col('cart_users')) * 100, 2)
    ).otherwise(0)
).withColumn(
    'overall_conversion',
    F.when(F.col('viewers') > 0,
           F.round((F.col('buyers') / F.col('viewers')) * 100, 2)
    ).otherwise(0)
)

(funnel_analysis.write.mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/conversion_funnel'))

funnel_analysis.show(truncate=False)

print(f'\n✅ Enhanced analytics tables saved to {ANALYTICS_STORAGE}')
print('\n💡 Business Insights:')
print('   • Focus on user behavior patterns (not product names)')
print('   • Cart abandonment rate is the key metric')
print('   • Conversion funnel shows where users drop off')
print('   • These metrics drive retention strategies')

📊 Creating enhanced analytics tables...

1️⃣ Event type distribution
+-----------+-------+----------+
|event      |count  |percentage|
+-----------+-------+----------+
|view       |2664218|96.68     |
|addtocart  |68966  |2.5       |
|transaction|22457  |0.81      |
+-----------+-------+----------+


2️⃣ Top items by engagement (with conversion metrics)


+------+------------+-----+-----+---------+------------+-----------------+---------------------+-----------------------+
|itemid|total_events|views|carts|purchases|unique_users|view_to_cart_rate|cart_to_purchase_rate|overall_conversion_rate|
+------+------------+-----+-----+---------+------------+-----------------+---------------------+-----------------------+
|187946|3412        |3410 |2    |0        |2912        |6.0E-4           |0.0                  |0.0                    |
|461686|2975        |2538 |304  |133      |1393        |0.1198           |0.4375               |0.0524                 |
|5411  |2334        |2325 |9    |0        |2079        |0.0039           |0.0                  |0.0                    |
|370653|1854        |1854 |0    |0        |1577        |0.0              |0.0                  |0.0                    |
|219512|1800        |1740 |48   |12       |1370        |0.0276           |0.25                 |0.0069                 |
|257040|1647        |1531 |89   

+----------+------------+------------+-----+-----+------------+---------------------+
|event_date|total_events|active_users|views|carts|transactions|daily_conversion_rate|
+----------+------------+------------+-----+-----+------------+---------------------+
|2015-05-03|13682       |7865        |13303|296  |83          |0.62                 |
|2015-05-04|19413       |11019       |18681|578  |154         |0.82                 |
|2015-05-05|23011       |13084       |22225|561  |225         |1.01                 |
|2015-05-06|23920       |13687       |23015|647  |258         |1.12                 |
|2015-05-07|23157       |13086       |22368|572  |217         |0.97                 |
|2015-05-08|19423       |11106       |18766|531  |126         |0.67                 |
|2015-05-09|12695       |7610        |12353|279  |63          |0.51                 |
|2015-05-10|12141       |7413        |11771|310  |60          |0.51                 |
|2015-05-11|16174       |9512        |15702|388  |84  

+-------+----------+------+-----------------------+---------------------------+------------------+
|viewers|cart_users|buyers|view_to_cart_conversion|cart_to_purchase_conversion|overall_conversion|
+-------+----------+------+-----------------------+---------------------------+------------------+
|1404179|37722     |11719 |2.69                   |31.07                      |0.83              |
+-------+----------+------+-----------------------+---------------------------+------------------+


✅ Enhanced analytics tables saved to /kaggle/working/ecommerce_bigdata/analytics

💡 Business Insights:
   • Focus on user behavior patterns (not product names)
   • Cart abandonment rate is the key metric
   • Conversion funnel shows where users drop off
   • These metrics drive retention strategies


---
##  Machine Learning (Spark MLlib)

**Tool:** Spark MLlib  
**Why:** Distributed ML training, scales with data  
**Alternative Rejected:** scikit-learn (single-machine, doesn't scale)

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

print('🤖 Preparing ML pipeline...\n')

# Load processed data
ml_data = spark.read.parquet(f'{PROCESSED_STORAGE}/user_features_parquet')

# Select features (numeric only)
feature_cols = [
    'view_count', 'addtocart_count', 'transaction_count',
    'unique_items_viewed', 'unique_items_carted', 'unique_items_purchased',
    'active_days', 'days_active_span', 'events_per_day',
    'view_to_cart_rate', 'cart_to_transaction_rate', 'view_to_transaction_rate',
    'has_cart_abandonment', 'is_browser_only', 'is_buyer',
    'engagement_score', 'total_events'
]

print(f'Features selected: {len(feature_cols)}')
print(f'Target: churn_risk_score (0=low, 1=medium, 2=high)')

🤖 Preparing ML pipeline...

Features selected: 17
Target: churn_risk_score (0=low, 1=medium, 2=high)


In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

print('🤖 Preparing ML pipeline...\n')

# Load processed data
ml_data = spark.read.parquet(f'{PROCESSED_STORAGE}/user_features_parquet')

# Safer features to reduce data leakage
feature_cols = [
    'view_count',
    'unique_items_viewed',
    'active_days',
    'days_active_span',
    'events_per_day',
    'engagement_score'
]

print(f'Features selected: {len(feature_cols)}')
print(f'Target: churn_risk_score (0=low, 1=medium, 2=high)')

🤖 Preparing ML pipeline...

Features selected: 6
Target: churn_risk_score (0=low, 1=medium, 2=high)


In [ ]:
ml_data = ml_data.withColumn(
    'churn_risk_score',
    F.when(F.col('churn_risk') == 'high', 2)
     .when(F.col('churn_risk') == 'medium', 1)
     .when(F.col('churn_risk') == 'low', 0)
)

In [ ]:
# Prepare data
ml_data_clean = (ml_data
                 .filter(F.col('churn_risk_score') >= 0)
                 .select(['visitorid'] + feature_cols + ['churn_risk_score', 'churn_risk'])
                 .fillna(0))

# Check class distribution
print('\n📊 Target distribution:')
ml_data_clean.groupBy('churn_risk', 'churn_risk_score').count().orderBy('churn_risk_score').show()

# Feature vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features_raw')
scaler = StandardScaler(inputCol='features_raw', outputCol='features',
                       withStd=True, withMean=True)

ml_data_vec = assembler.transform(ml_data_clean)
scaler_model = scaler.fit(ml_data_vec)
ml_data_scaled = scaler_model.transform(ml_data_vec)
ml_data_final = ml_data_scaled.withColumnRenamed('churn_risk_score', 'label')

# Train/test split
train_data, test_data = ml_data_final.randomSplit([0.8, 0.2], seed=42)

train_count = train_data.count()
test_count = test_data.count()

print(f'\n✅ ML data prepared')
print(f'   Train: {train_count:,} samples')
print(f'   Test:  {test_count:,} samples')


📊 Target distribution:
+----------+----------------+------+
|churn_risk|churn_risk_score| count|
+----------+----------------+------+
|       low|               0|  7414|
|    medium|               1|106465|
|      high|               2| 16897|
+----------+----------------+------+


✅ ML data prepared
   Train: 104,764 samples
   Test:  26,012 samples


In [ ]:
# Train models
print('\n🤖 Training models...\n')

evaluator = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction')

# 1. Logistic Regression
print('1️⃣ Logistic Regression')
lr = LogisticRegression(maxIter=100, regParam=0.01)
lr_model = lr.fit(train_data)
lr_pred = lr_model.transform(test_data)

lr_acc = evaluator.evaluate(lr_pred, {evaluator.metricName: 'accuracy'})
lr_f1 = evaluator.evaluate(lr_pred, {evaluator.metricName: 'f1'})
print(f'   Accuracy: {lr_acc:.4f}  F1: {lr_f1:.4f}')

# 2. Decision Tree
print('\n2️⃣ Decision Tree')
dt = DecisionTreeClassifier(maxDepth=10, minInstancesPerNode=10)
dt_model = dt.fit(train_data)
dt_pred = dt_model.transform(test_data)

dt_acc = evaluator.evaluate(dt_pred, {evaluator.metricName: 'accuracy'})
dt_f1 = evaluator.evaluate(dt_pred, {evaluator.metricName: 'f1'})
print(f'   Accuracy: {dt_acc:.4f}  F1: {dt_f1:.4f}')

# Feature importance
print('\n   Top 5 important features:')
fi = [(feature_cols[i], float(dt_model.featureImportances[i]))
      for i in range(len(feature_cols))]
fi_sorted = sorted(fi, key=lambda x: x[1], reverse=True)
for feat, imp in fi_sorted[:5]:
    print(f'     {feat:30s} {imp:.4f}')

# 3. Random Forest
print('\n3️⃣ Random Forest')
rf = RandomForestClassifier(numTrees=50, maxDepth=10, seed=42)
rf_model = rf.fit(train_data)
rf_pred = rf_model.transform(test_data)

rf_acc = evaluator.evaluate(rf_pred, {evaluator.metricName: 'accuracy'})
rf_f1 = evaluator.evaluate(rf_pred, {evaluator.metricName: 'f1'})
print(f'   Accuracy: {rf_acc:.4f}  F1: {rf_f1:.4f}')


🤖 Training models...

1️⃣ Logistic Regression
   Accuracy: 0.8169  F1: 0.7385

2️⃣ Decision Tree
   Accuracy: 0.9845  F1: 0.9844

   Top 5 important features:
     view_count                     0.6625
     engagement_score               0.2998
     unique_items_viewed            0.0247
     events_per_day                 0.0100
     active_days                    0.0023

3️⃣ Random Forest


   Accuracy: 0.9854  F1: 0.9855


In [ ]:
# Model comparison
import pandas as pd

print('\n' + '=' * 70)
print('MODEL COMPARISON')
print('=' * 70 + '\n')

comparison_df = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Accuracy': lr_acc, 'F1': lr_f1},
    {'Model': 'Decision Tree', 'Accuracy': dt_acc, 'F1': dt_f1},
    {'Model': 'Random Forest', 'Accuracy': rf_acc, 'F1': rf_f1}
])

print(comparison_df.to_string(index=False))

# Best model
best_idx = comparison_df['F1'].idxmax()
best_name = comparison_df.loc[best_idx, 'Model']
best_f1 = comparison_df.loc[best_idx, 'F1']

print(f'\n🏆 BEST MODEL: {best_name} (F1={best_f1:.4f})')

# Save comparison
comparison_spark = spark.createDataFrame(comparison_df)
(comparison_spark.write.mode('overwrite')
 .parquet(f'{ANALYTICS_STORAGE}/model_comparison'))

print(f'\n✅ Model comparison saved')

# Select best model
if best_name == 'Random Forest':
    best_model, best_pred = rf_model, rf_pred
elif best_name == 'Decision Tree':
    best_model, best_pred = dt_model, dt_pred
else:
    best_model, best_pred = lr_model, lr_pred

# Save best model
best_model.write().overwrite().save(f'{ML_STORAGE}/best_churn_model')
print(f'✅ Best model saved to {ML_STORAGE}/best_churn_model')


MODEL COMPARISON

              Model  Accuracy       F1
Logistic Regression  0.816931 0.738536
      Decision Tree  0.984507 0.984369
      Random Forest  0.985353 0.985495

🏆 BEST MODEL: Random Forest (F1=0.9855)

✅ Model comparison saved
✅ Best model saved to /kaggle/working/ecommerce_bigdata/ml/best_churn_model


---
##  Decision Engine & Retention Actions

In [ ]:
print('🎯 Generating personalized retention actions...\n')

# Map predictions to risk
final_decisions = best_pred.withColumn(
    'predicted_risk',
    F.when(F.col('prediction') == 2, 'high')
     .when(F.col('prediction') == 1, 'medium')
     .otherwise('low')
)

# Add recommended actions
final_decisions = final_decisions.withColumn(
    'recommended_action',
    F.when(F.col('predicted_risk') == 'high',
           'Send 20% discount code + cart expiry reminder (24h)')
     .when(F.col('predicted_risk') == 'medium',
           'Send personalized product recommendations + 10% first-purchase discount')
     .otherwise('Maintain engagement via newsletters and new arrivals')
)

# Priority
final_decisions = final_decisions.withColumn(
    'priority',
    F.when(F.col('predicted_risk') == 'high', 1)
     .when(F.col('predicted_risk') == 'medium', 2)
     .otherwise(3)
)

print(f'✅ Generated {final_decisions.count():,} personalized decisions')

# Sample decisions
print('\n📋 Sample retention actions:\n')
final_decisions.select(
    'predicted_risk', 'priority', 'recommended_action'
).orderBy('priority').show(5, truncate=60)

🎯 Generating personalized retention actions...

✅ Generated 26,012 personalized decisions

📋 Sample retention actions:

+--------------+--------+---------------------------------------------------+
|predicted_risk|priority|                                 recommended_action|
+--------------+--------+---------------------------------------------------+
|          high|       1|Send 20% discount code + cart expiry reminder (24h)|
|          high|       1|Send 20% discount code + cart expiry reminder (24h)|
|          high|       1|Send 20% discount code + cart expiry reminder (24h)|
|          high|       1|Send 20% discount code + cart expiry reminder (24h)|
|          high|       1|Send 20% discount code + cart expiry reminder (24h)|
+--------------+--------+---------------------------------------------------+
only showing top 5 rows



In [ ]:
# Business impact estimation
print('=' * 70)
print('ESTIMATED BUSINESS IMPACT')
print('=' * 70)

high_risk_users = final_decisions.filter(F.col('predicted_risk') == 'high').count()

# Assume 15% conversion rate from discount campaign
expected_conversions = high_risk_users * 0.15

print(f'\nHigh-risk users targeted:  {high_risk_users:>12,}')
print(f'Expected conversion rate:  {15:>12}%')
print(f'Potential conversions:     {expected_conversions:>12,.0f}')
print('\n💰 This system enables:')
print('   • Proactive retention (vs reactive)')
print('   • Personalized actions (vs mass campaigns)')
print('   • Data-driven decisions (vs gut feeling)')
print('   • Measurable ROI (vs unmeasured spend)')
print('=' * 70)

ESTIMATED BUSINESS IMPACT

High-risk users targeted:         3,141
Expected conversion rate:            15%
Potential conversions:              471

💰 This system enables:
   • Proactive retention (vs reactive)
   • Personalized actions (vs mass campaigns)
   • Data-driven decisions (vs gut feeling)
   • Measurable ROI (vs unmeasured spend)


---
##  CSV Exports for Dashboards

In [ ]:
print('📤 Exporting analytics to CSV for dashboards...\n')

tables_to_export = [
    'event_type_distribution',
    'top_items',
    'daily_activity',
    'churn_risk_distribution',
    'cart_abandonment',
    'model_comparison'
]

for table in tables_to_export:
    df_export = spark.read.parquet(f'{ANALYTICS_STORAGE}/{table}')
    (df_export.coalesce(1).write
     .mode('overwrite')
     .option('header', 'true')
     .csv(f'{EXPORT_STORAGE}/{table}'))
    print(f'  ✅ {table}.csv')

# Export top 1000 priority decisions
print('\n  ✅ final_user_decisions.csv (top 1000)')
(final_decisions
 .select('visitorid', 'predicted_risk', 'priority', 'recommended_action')
 .orderBy('priority')
 .limit(1000)
 .coalesce(1).write
 .mode('overwrite')
 .option('header', 'true')
 .csv(f'{EXPORT_STORAGE}/final_user_decisions'))

print(f'\n✅ All exports complete')
print(f'   Location: {EXPORT_STORAGE}/')
print('\n💡 These CSVs are ready for:')
print('   • Tableau / Power BI dashboards')
print('   • Excel analysis')
   • Marketing team CRM integration')
print('   • Email campaign systems')

IndentationError: unexpected indent (2397677251.py, line 36)

In [ ]:
print('📤 Exporting analytics to CSV for dashboards...\n')

tables_to_export = [
    'event_type_distribution',
    'top_items',
    'daily_activity',
    'churn_risk_distribution',
    'cart_abandonment',
    'model_comparison'
]

for table in tables_to_export:
    df_export = spark.read.parquet(f'{ANALYTICS_STORAGE}/{table}')

    (df_export.coalesce(1).write
     .mode('overwrite')
     .option('header', 'true')
     .csv(f'{EXPORT_STORAGE}/{table}'))

    print(f'  ✅ {table}.csv')

# Export top 1000 priority decisions
print('\n  ✅ final_user_decisions.csv (top 1000)')

(final_decisions
 .select('visitorid', 'predicted_risk', 'priority', 'recommended_action')
 .orderBy('priority')
 .limit(1000)
 .coalesce(1).write
 .mode('overwrite')
 .option('header', 'true')
 .csv(f'{EXPORT_STORAGE}/final_user_decisions'))

print(f'\n✅ All exports complete')
print(f'   Location: {EXPORT_STORAGE}/')

print('\n💡 These CSVs are ready for:')
print('   • Tableau / Power BI dashboards')
print('   • Excel analysis')
print('   • Marketing team CRM integration')
print('   • Email campaign systems')

📤 Exporting analytics to CSV for dashboards...

  ✅ event_type_distribution.csv
  ✅ top_items.csv
  ✅ daily_activity.csv
  ✅ churn_risk_distribution.csv
  ✅ cart_abandonment.csv
  ✅ model_comparison.csv

  ✅ final_user_decisions.csv (top 1000)

✅ All exports complete
   Location: /kaggle/working/ecommerce_bigdata/exports/

💡 These CSVs are ready for:
   • Tableau / Power BI dashboards
   • Excel analysis
   • Marketing team CRM integration
   • Email campaign systems


In [ ]:
import pandas as pd

df = spark.read.parquet(f'{ANALYTICS_STORAGE}/event_type_distribution').toPandas()
df.to_csv(f'{EXPORT_STORAGE}/event_type_distribution.csv', index=False)

In [ ]:
import pandas as pd

tables_to_export = [
    'event_type_distribution',
    'top_items',
    'daily_activity',
    'churn_risk_distribution',
    'cart_abandonment',
    'model_comparison'
]

for table in tables_to_export:
    df = spark.read.parquet(f'{ANALYTICS_STORAGE}/{table}')

    pdf = df.toPandas()  # تحويل إلى pandas

    file_path = f'{EXPORT_STORAGE}/{table}.csv'
    pdf.to_csv(file_path, index=False)

    print(f'✅ Saved: {file_path}')

✅ Saved: /kaggle/working/ecommerce_bigdata/exports/event_type_distribution.csv
✅ Saved: /kaggle/working/ecommerce_bigdata/exports/top_items.csv
✅ Saved: /kaggle/working/ecommerce_bigdata/exports/daily_activity.csv
✅ Saved: /kaggle/working/ecommerce_bigdata/exports/churn_risk_distribution.csv
✅ Saved: /kaggle/working/ecommerce_bigdata/exports/cart_abandonment.csv
✅ Saved: /kaggle/working/ecommerce_bigdata/exports/model_comparison.csv


In [ ]:
pdf = (final_decisions
       .select('visitorid', 'predicted_risk', 'priority', 'recommended_action')
       .orderBy('priority')
       .limit(1000)
       .toPandas())

pdf.to_csv(f'{EXPORT_STORAGE}/final_user_decisions.csv', index=False)

print('✅ Saved: final_user_decisions.csv')

✅ Saved: final_user_decisions.csv
